# Training the EN -> Urdu Transformer

This notebook trains the from-scratch Transformer on your cleaned OPUS data.

Sized for a 6GB laptop GPU (RTX 3050): a trimmed model config, mixed-precision
(AMP) training, gradient accumulation as a memory safety valve, and
checkpointing so a training run can resume if interrupted.

Run this from the same folder as `transformer.py` and `dataset.py`.


In [ ]:
import os
import time
import math
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
import sentencepiece as spm
import matplotlib.pyplot as plt

from transformer import Transformer
from dataset import build_dataloader, PAD_IDX, SOS_IDX, EOS_IDX

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Config

Sized down from the paper's "base" config (d_model=512, 6 layers) to fit
comfortably in 6GB VRAM with room to spare for activations and gradients.
If you have VRAM to spare after a first run, `d_model` and `num_layers` are
the first knobs to raise.


In [ ]:
CONFIG = {
    # model size -- trimmed for 6GB VRAM
    "d_model": 256,
    "num_layers": 4,
    "num_heads": 8,
    "d_ff": 1024,
    "dropout": 0.1,
    "max_len": 100,

    # training
    "max_tokens_per_batch": 4000,   # token-count batching, not fixed batch size
    "num_epochs": 20,
    "label_smoothing": 0.1,
    "grad_clip": 1.0,
    "warmup_steps": 4000,           # LR warmup as in the original paper
    "grad_accum_steps": 2,          # effective batch size = 2x, at half the peak VRAM

    # paths
    "train_en": "data/processed/train.en",
    "train_ur": "data/processed/train.ur",
    "val_fraction": 0.01,           # held-out slice of train data for validation
    "val_seed": 42,
    "tokenizer_en": "tokenizers/en.model",
    "tokenizer_ur": "tokenizers/ur.model",
    "checkpoint_dir": "checkpoints",
    "checkpoint_every_steps": 2000,
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
CONFIG


## Train / validation split

The data pipeline so far only produced one combined file. We hold out a
small random slice here for validation -- enough to track whether the
model is generalizing, without giving up much training data.


In [ ]:
import random

def make_val_split():
    with open(CONFIG["train_en"], encoding="utf-8") as f:
        en_lines = [l.strip() for l in f if l.strip()]
    with open(CONFIG["train_ur"], encoding="utf-8") as f:
        ur_lines = [l.strip() for l in f if l.strip()]

    assert len(en_lines) == len(ur_lines)

    rng = random.Random(CONFIG["val_seed"])
    indices = list(range(len(en_lines)))
    rng.shuffle(indices)

    n_val = max(1, int(len(indices) * CONFIG["val_fraction"]))
    val_idx = set(indices[:n_val])

    train_en, train_ur, val_en, val_ur = [], [], [], []
    for i, (en, ur) in enumerate(zip(en_lines, ur_lines)):
        if i in val_idx:
            val_en.append(en)
            val_ur.append(ur)
        else:
            train_en.append(en)
            train_ur.append(ur)

    os.makedirs("data/split", exist_ok=True)
    with open("data/split/train.en", "w", encoding="utf-8") as f:
        f.write("\n".join(train_en))
    with open("data/split/train.ur", "w", encoding="utf-8") as f:
        f.write("\n".join(train_ur))
    with open("data/split/val.en", "w", encoding="utf-8") as f:
        f.write("\n".join(val_en))
    with open("data/split/val.ur", "w", encoding="utf-8") as f:
        f.write("\n".join(val_ur))

    print(f"train: {len(train_en)} pairs, val: {len(val_en)} pairs")

make_val_split()


In [ ]:
sp_en = spm.SentencePieceProcessor(model_file=CONFIG["tokenizer_en"])
sp_ur = spm.SentencePieceProcessor(model_file=CONFIG["tokenizer_ur"])

train_loader = build_dataloader(
    "data/split/train.en", "data/split/train.ur",
    sp_en, sp_ur, max_tokens=CONFIG["max_tokens_per_batch"], shuffle=True,
)
val_loader = build_dataloader(
    "data/split/val.en", "data/split/val.ur",
    sp_en, sp_ur, max_tokens=CONFIG["max_tokens_per_batch"], shuffle=False,
)

print(f"~{len(train_loader)} train batches/epoch, ~{len(val_loader)} val batches")


## Model, optimizer, LR schedule

The warmup + inverse-square-root LR schedule from the original paper --
LR ramps up linearly for `warmup_steps`, then decays proportional to
`1/sqrt(step)`. This is more stable early in training than a flat LR,
especially for Transformers (they're notoriously sensitive to the first
few thousand steps).


In [ ]:
src_vocab_size = sp_en.get_piece_size()
tgt_vocab_size = sp_ur.get_piece_size()
print(f"src_vocab={src_vocab_size}, tgt_vocab={tgt_vocab_size}")

model = Transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=CONFIG["d_model"],
    num_layers=CONFIG["num_layers"],
    num_heads=CONFIG["num_heads"],
    d_ff=CONFIG["d_ff"],
    max_len=CONFIG["max_len"],
    dropout=CONFIG["dropout"],
    pad_idx=PAD_IDX,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")


In [ ]:
def lr_lambda(step):
    step = max(step, 1)
    d_model = CONFIG["d_model"]
    warmup = CONFIG["warmup_steps"]
    return (d_model ** -0.5) * min(step ** -0.5, step * warmup ** -1.5)

# base lr=1.0 here because lr_lambda already computes the full scaled value
optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=CONFIG["label_smoothing"])

# AMP: scales gradients to avoid underflow in float16, critical for stable
# mixed-precision training -- and mixed precision roughly halves VRAM use,
# which matters a lot on a 6GB card
scaler = GradScaler(device="cuda", enabled=(device.type == "cuda"))


## Checkpointing

Saves model/optimizer/scheduler state plus the step count, so a training
run can be killed and resumed without starting over -- important for long
runs on a laptop that might sleep, need a reboot, etc.


In [ ]:
def save_checkpoint(step, epoch, path):
    torch.save({
        "step": step,
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "config": CONFIG,
    }, path)

def load_checkpoint(path):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    scaler.load_state_dict(ckpt["scaler_state"])
    print(f"Resumed from step {ckpt['step']}, epoch {ckpt['epoch']}")
    return ckpt["step"], ckpt["epoch"]

# to resume a previous run, uncomment and point at your checkpoint:
# global_step, start_epoch = load_checkpoint("checkpoints/latest.pt")
global_step, start_epoch = 0, 0


## Training loop

One epoch = one full pass over the training data. Gradient accumulation
splits an "effective" batch into `grad_accum_steps` smaller forward/backward
passes before stepping the optimizer -- this keeps peak VRAM usage down
while still getting the benefit of a larger effective batch size.


In [ ]:
train_losses, val_losses = [], []

def run_validation():
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for src, tgt_in, tgt_out in val_loader:
            src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
            with autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logits = model(src, tgt_in)
                loss = criterion(logits.reshape(-1, tgt_vocab_size), tgt_out.reshape(-1))
            n_tok = (tgt_out != PAD_IDX).sum().item()
            total_loss += loss.item() * n_tok
            total_tokens += n_tok
    model.train()
    return total_loss / max(total_tokens, 1)


for epoch in range(start_epoch, CONFIG["num_epochs"]):
    model.train()
    epoch_start = time.time()
    running_loss, running_tokens = 0.0, 0

    optimizer.zero_grad()
    for i, (src, tgt_in, tgt_out) in enumerate(train_loader):
        src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)

        with autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, tgt_vocab_size), tgt_out.reshape(-1))
            loss = loss / CONFIG["grad_accum_steps"]

        scaler.scale(loss).backward()

        if (i + 1) % CONFIG["grad_accum_steps"] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % CONFIG["checkpoint_every_steps"] == 0:
                save_checkpoint(global_step, epoch, os.path.join(CONFIG["checkpoint_dir"], "latest.pt"))

        n_tok = (tgt_out != PAD_IDX).sum().item()
        running_loss += loss.item() * CONFIG["grad_accum_steps"] * n_tok
        running_tokens += n_tok

        if i % 200 == 0:
            avg_loss = running_loss / max(running_tokens, 1)
            current_lr = scheduler.get_last_lr()[0]
            print(f"epoch {epoch} step {i}/{len(train_loader)} "
                  f"loss {avg_loss:.4f} lr {current_lr:.2e}")

    train_loss = running_loss / max(running_tokens, 1)
    val_loss = run_validation()
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    elapsed = time.time() - epoch_start
    print(f"== epoch {epoch} done in {elapsed/60:.1f} min -- "
          f"train_loss {train_loss:.4f}, val_loss {val_loss:.4f} ==")

    save_checkpoint(global_step, epoch + 1,
                     os.path.join(CONFIG["checkpoint_dir"], f"epoch_{epoch}.pt"))
    save_checkpoint(global_step, epoch + 1,
                     os.path.join(CONFIG["checkpoint_dir"], "latest.pt"))


## Loss curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="train loss")
plt.plot(val_losses, label="val loss")
plt.xlabel("epoch")
plt.ylabel("cross-entropy loss (per token)")
plt.legend()
plt.title("Training progress")
plt.show()


## Try a translation

Beam search decoding (fixes the repetition-loop issue greedy decoding had)
on a sentence of your choice, to eyeball quality as you go.


In [ ]:
def translate(sentence: str, max_len: int = 60, beam_size: int = 5) -> str:
    model.eval()
    src_ids = sp_en.encode(sentence, out_type=int)
    src = torch.tensor([src_ids], dtype=torch.long, device=device)

    out_ids = model.beam_search_decode(
        src, sos_idx=SOS_IDX, eos_idx=EOS_IDX,
        beam_size=beam_size, max_len=max_len, no_repeat_ngram_size=3,
    )
    out_ids = out_ids[0].tolist()

    if EOS_IDX in out_ids:
        out_ids = out_ids[1:out_ids.index(EOS_IDX)]
    else:
        out_ids = out_ids[1:]

    return sp_ur.decode(out_ids)

print(translate("Hello, how are you?"))
print(translate("This is a test sentence."))


## Real evaluation: IN22-Conv

Two sample sentences don't tell you much. IN22-Conv (AI4Bharat) is a
1,503-sentence conversational-domain benchmark, held out entirely from
training -- this gives an honest read on everyday-conversation quality
instead of anecdotal spot checks.

Requires `pip install sacrebleu datasets` and being logged in via
`hf auth login` (same token as before, same dataset terms accepted).


In [ ]:
from datasets import load_dataset
import sacrebleu

def load_in22_conv():
    ds = load_dataset("ai4bharat/IN22-Conv", split="test")
    en_sents = list(ds["eng_Latn"])
    ur_refs = list(ds["urd_Arab"])
    return en_sents, ur_refs

en_sents, ur_refs = load_in22_conv()
print(f"Loaded {len(en_sents)} conversational eval sentences")


In [ ]:
# translate the eval set -- this will take a while with beam search over
# 1503 sentences on a laptop GPU, that's expected
hypotheses = []
for i, sent in enumerate(en_sents):
    hyp = translate(sent, beam_size=4)  # smaller beam here purely for eval speed
    hypotheses.append(hyp)
    if i % 100 == 0:
        print(f"{i}/{len(en_sents)}")

bleu = sacrebleu.corpus_bleu(hypotheses, [ur_refs])
print(f"\nBLEU on IN22-Conv: {bleu.score:.2f}")

print("\nA few examples:")
for i in [0, 1, 2, 3, 4]:
    print(f"  EN:  {en_sents[i]}")
    print(f"  REF: {ur_refs[i]}")
    print(f"  HYP: {hypotheses[i]}")
    print()
